# Building a Chatbot with RAG from Scratch 🤖📚
### A Hands-On Guide to Retrieval-Augmented Generation, Agents, and Hallucination Prevention

**Workshop:** NTIR · **Presenter:** Hakan Tugrul Otal *(PhD Student, Dept. of Information Sciences & Technology, UAlbany)*

---

### What we'll build today

1. A **local LLM chatbot** (no cloud API)
2. A **document pipeline**: load → chunk → embed → store
3. A **simple RAG chain** with anti-hallucination prompting
4. An **Agentic RAG** system using CrewAI (Retriever + Verifier)
5. A **Gradio chat UI** you can share with a public link
6. A **head-to-head comparison**: Plain LLM vs Simple RAG vs Agentic RAG

### How to run this

- **Runtime → Run all** (or just execute cells top-to-bottom)
- GPU is **not required** but helpful. To enable: *Runtime → Change runtime type → T4 GPU*
- Setup (Section 0) takes ~2-3 minutes — start it first, then read ahead

## Section 0 — Setup

We install Ollama (a runtime for running open-source LLMs locally), pull two small models,
install Python dependencies, and download *Frankenstein*.

🕐 **Timing:** ~2-3 minutes total. Kick this off and wait.

In [1]:
# Sanity check
import sys, subprocess
assert sys.version_info >= (3, 10), "Python 3.10+ required"
try:
    gpu = subprocess.run(['nvidia-smi'], capture_output=True, text=True, timeout=5)
    print("✅ GPU available" if gpu.returncode == 0 else "⚠️  No GPU — running on CPU (fine, just slower)")
except Exception:
    print("⚠️  No GPU — running on CPU (fine, just slower)")
print(f"✅ Python {sys.version.split()[0]}")

✅ GPU available
✅ Python 3.12.13


In [2]:
# Install Ollama (the LLM runtime). ~30 seconds.
!apt-get install -y zstd pciutils
!curl -fsSL https://ollama.com/install.sh | sh 2>&1 | tail -3

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libpci3 pci.ids
The following NEW packages will be installed:
  libpci3 pci.ids pciutils zstd
0 upgraded, 4 newly installed, 0 to remove and 42 not upgraded.
Need to get 946 kB of archives.
After this operation, 3,276 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 pci.ids all 0.0~2022.01.22-1ubuntu0.1 [251 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/main amd64 libpci3 amd64 1:3.7.0-6 [28.9 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy/main amd64 pciutils amd64 1:3.7.0-6 [63.6 kB]
Get:4 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 946 kB in 2s (469 kB/s)
Selecting previously unselected package pci.ids.
(Reading database ... 122354 files and directories currently installed.)
Preparing to unpack .../pci.ids_0.0~2022.01.22-

In [3]:
# Start the Ollama server as a background process
import os, subprocess, time, requests

os.environ['OLLAMA_HOST'] = '127.0.0.1:11434'
subprocess.Popen(
    ['ollama', 'serve'],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

# Wait for the server to be ready
for i in range(20):
    try:
        if requests.get('http://127.0.0.1:11434', timeout=1).ok:
            print(f"✅ Ollama server running (took {i+1}s)")
            break
    except Exception:
        time.sleep(1)
else:
    print("⚠️  Ollama server did not start — try running this cell again")

✅ Ollama server running (took 2s)


In [ ]:
# Pull the models. llama3.2:3b is a solid small model (~2GB) — good quality, fast on CPU.
# If you're on a constrained runtime, swap for llama3.2:1b (smaller, faster, less accurate).
print("Pulling llama3.2:3b (~2GB)...")
!ollama pull llama3.2:3b

print("\nPulling nomic-embed-text (~270MB)...")
!ollama pull nomic-embed-text

print("\n✅ Models ready:")
!ollama list

In [5]:
# Install Python dependencies
# (Versions pinned to known-working combinations as of the workshop date)
%pip install -q --upgrade \
    langchain \
    langchain-community \
    langchain-ollama \
    langchain-chroma \
    langchain_text_splitters \
    chromadb \
    crewai \
    gradio \
    pypdf \
    pandas

print("✅ Dependencies installed")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 7.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 6.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 5.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.9/67.9 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 102.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 118.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 431.6/431.6 kB 40.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 139.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.4/16.4 MB 98.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.6/19.6 MB 123.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.6/59.6 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33

In [6]:
# Download Frankenstein from Project Gutenberg
import urllib.request

url = "https://www.gutenberg.org/files/84/84-0.txt"
urllib.request.urlretrieve(url, "frankenstein.txt")

with open("frankenstein.txt", "r", encoding="utf-8") as f:
    text = f.read()

print(f"✅ Downloaded {len(text):,} characters")
print(f"\nPreview (skipping Gutenberg header):")
print(text[1400:1800])

✅ Downloaded 419,434 characters

Preview (skipping Gutenberg header):
light. There, Margaret, the sun is for ever
visible, its broad disk just skirting the horizon and diffusing a
perpetual splendour. There—for with your leave, my sister, I will put
some trust in preceding navigators—there snow and frost are banished;
and, sailing over a calm sea, we may be wafted to a land surpassing in
wonders and in beauty every region hitherto discovered on the habitable
globe. 


---

## Section 1 — Talk to the LLM (no RAG yet)

Before we build anything fancy, let's see what the raw LLM knows about Frankenstein.

🔮 **Predict first:** what will a small 1B-parameter model say is the **name of the monster**?

In [ ]:
import ollama

def ask_plain_llm(question, model='llama3.2:3b'):
    '''Ask the LLM with no context, no retrieval — just the question.'''
    response = ollama.chat(
        model=model,
        messages=[{'role': 'user', 'content': question}],
        options={'temperature': 0},
    )
    return response['message']['content']

# The classic trick question
q = "What is the name of the monster in Mary Shelley's Frankenstein?"
print(f"Q: {q}\n")
print(f"A: {ask_plain_llm(q)}")

💡 **Observation:** In the novel, **the monster has no name.** Small LLMs often confidently say "Frankenstein" — that's the name of the *scientist*, not the creature.

This is **hallucination**: plausible-sounding text that isn't true. It happens because the LLM pattern-matches popular culture rather than the actual book.

**Fix:** give the model the real text to read. That's RAG.

---

## Section 2 — Load and chunk the book

We can't just stuff the whole book into a prompt (LLMs have context limits, and it's wasteful).
Instead, we split it into ~800-character chunks with some overlap, so we can retrieve only the relevant pieces later.

**Why these numbers?**
- `chunk_size=800` — big enough for ~2 paragraphs, small enough for precise retrieval
- `chunk_overlap=100` — prevents key sentences from being cut in half at chunk boundaries
- `separators=["\n\n", "\n", ". ", " "]` — prefer to split at paragraph → sentence → word boundaries

In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

# Load the book
with open("frankenstein.txt", "r", encoding="utf-8") as f:
    raw = f.read()

# Strip Project Gutenberg's legal header/footer
start_marker = "Letter 1"
end_marker = "*** END OF THE PROJECT GUTENBERG"
start = raw.find(start_marker)
end = raw.find(end_marker)
book = raw[start:end] if start > 0 and end > 0 else raw

# Split into overlapping chunks
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100,
    separators=["\n\n", "\n", ". ", " ", ""],
)
chunks = [Document(page_content=c) for c in splitter.split_text(book)]

print(f"📖 Book: {len(book):,} characters → {len(chunks)} chunks")
print(f"\n--- Chunk #50 (sample) ---")
print(chunks[50].page_content[:500])

📖 Book: 419,243 characters → 748 chunks

--- Chunk #50 (sample) ---
He then told me that he would commence his narrative the next day when I
should be at leisure. This promise drew from me the warmest thanks. I have
resolved every night, when I am not imperatively occupied by my duties, to
record, as nearly as possible in his own words, what he has related during
the day. If I should be engaged, I will at least make notes. This
manuscript will doubtless afford you the greatest pleasure; but to me, who
know him, and who hear it from his own lips—with what interes


---

## Section 3 — Embed and store in a vector database

Now we turn each chunk into a **vector** (a list of numbers that captures its meaning) using `nomic-embed-text`.
Similar meanings → similar vectors → we can find relevant chunks by similarity search.

We store everything in **ChromaDB**, a lightweight local vector database.

🕐 **Timing:** ~30-60 seconds to embed all chunks on CPU.

In [9]:
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma
import shutil, os

# Clean slate (useful if re-running)
if os.path.exists("./chroma_db"):
    shutil.rmtree("./chroma_db")

embeddings = OllamaEmbeddings(model="nomic-embed-text")

# Build the vector store
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    # persist_directory="./chroma_db",
)

print(f"✅ Indexed {len(chunks)} chunks into ChromaDB")

✅ Indexed 748 chunks into ChromaDB


In [10]:
# Test: find passages semantically related to a question
# (note: we haven't called the LLM yet — this is pure retrieval)
query = "Does the monster have a name?"
results = vectorstore.similarity_search(query, k=3)

for i, r in enumerate(results, 1):
    print(f"--- Match {i} ---")
    print(r.page_content[:300])
    print()

--- Match 1 ---
His tale is connected and told with an appearance of the simplest truth,
yet I own to you that the letters of Felix and Safie, which he showed me,
and the apparition of the monster seen from our ship, brought to me a
greater conviction of the truth of his narrative than his asseverations,
however ea

--- Match 2 ---
yellow light of the moon, as it forced its way through the window
shutters, I beheld the wretch—the miserable monster whom I had
created. He held up the curtain of the bed; and his eyes, if eyes they
may be called, were fixed on me. His jaws opened, and he muttered some
inarticulate sounds, while a 

--- Match 3 ---
them; sometimes it was the watery, clouded eyes of the monster, as I
first saw them in my chamber at Ingolstadt.



💡 **Notice:** retrieval already "finds" passages that discuss how the creature is referred to in the book — without any LLM involvement. Retrieval and generation are separate concerns.

---

## Section 4 — Simple RAG chain

Time to wire it together:

```
question → retrieve top-k chunks → stuff into prompt → LLM → grounded answer
```

The **prompt** is our most important hallucination defense. We explicitly tell the model:
- Use **only** the provided context
- Say *"The text does not specify"* if the answer isn't there
- Do **not** use outside knowledge

This single instruction does more for accuracy than most fancy techniques.

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

llm = ChatOllama(model="llama3.2:3b", temperature=0)
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

RAG_PROMPT = ChatPromptTemplate.from_template('''You are a literary assistant answering questions about the novel Frankenstein by Mary Shelley.

Use ONLY the context below to answer. If the answer is not in the context, reply exactly:
"The text does not specify."
Do not use any outside knowledge. Do not speculate.

Context:
{context}

Question: {question}

Answer:''')

def format_docs(docs):
    return "\n\n---\n\n".join(d.page_content for d in docs)

simple_rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | RAG_PROMPT
    | llm
    | StrOutputParser()
)

print("✅ Simple RAG chain built")

In [12]:
# The payoff — same question as Section 1, now grounded in the actual book
q = "Does the monster have a name?"
print(f"Q: {q}\n")
print(f"A: {simple_rag_chain.invoke(q)}")

Q: Does the monster have a name?

A: The text does not specify.


🎯 **That's RAG.** The LLM didn't get smarter — we just gave it the right text to read.

Let's try a few more:

In [13]:
for question in [
    "Who rescues Victor from the Arctic ice?",
    "What books does the creature find and read in the forest?",
    "What is the capital of Switzerland?",  # ← not in the book — should refuse
]:
    print(f"Q: {question}")
    print(f"A: {simple_rag_chain.invoke(question)}\n")

Q: Who rescues Victor from the Arctic ice?
A: The text does not specify.

Q: What books does the creature find and read in the forest?
A: The text does not specify.

Q: What is the capital of Switzerland?
A: The text does not specify.



💡 **The "capital of Switzerland" example is important.** A plain LLM would happily answer "Bern" — a correct fact, but **out of scope**. Good RAG systems know when to stay in their lane.

---

## Section 5 — Agentic RAG with CrewAI

Simple RAG has a weakness: it trusts the first retrieval. If the top-k chunks miss key context, the answer suffers — and there's no double-check.

**Agentic RAG** introduces multiple LLM-powered agents that collaborate. Ours has two:

1. **🔍 Retriever Agent** — searches the book, drafts an answer with quotes
2. **✅ Verifier Agent** — reviews the draft, flags unsupported claims, produces the final answer

The verifier is the key insight. It acts like an editor demanding citations — it rejects anything not directly supported by the retrieved passages.

In [ ]:
from crewai import Agent, Task, Crew, Process, LLM
from crewai.tools import tool

# CrewAI wants a LiteLLM-style model string with provider prefix
crew_llm = LLM(
    model="ollama/llama3.2:3b",
    base_url="http://localhost:11434",
    temperature=0,
)

@tool("search_frankenstein")
def search_tool(query: str) -> str:
    """Searches the novel Frankenstein and returns the most relevant passages."""
    docs = vectorstore.similarity_search(query, k=4)
    return "\n\n---\n\n".join(d.page_content for d in docs)

retriever_agent = Agent(
    role="Literary Researcher",
    goal=("Find the most relevant passages from Frankenstein and draft a grounded "
          "answer with direct quotes from the text."),
    backstory=("You are a scholar of 19th century gothic literature. "
               "You always ground your claims in the actual text and quote passages verbatim."),
    tools=[search_tool],
    llm=crew_llm,        # ← the LiteLLM-wrapped one, NOT ChatOllama
    verbose=True,
    allow_delegation=False,
)

verifier_agent = Agent(
    role="Fact Verifier",
    goal=("Check that every claim in the draft answer is directly supported by "
          "the retrieved passages. Remove or correct unsupported claims."),
    backstory=("You are a skeptical editor who demands textual evidence for every claim. "
               "You are not afraid to say 'The text does not specify.'"),
    llm=crew_llm,
    verbose=True,
    allow_delegation=False,
)

print("✅ Agents defined")

In [19]:
def run_agentic_rag(question: str) -> str:
    '''Run the two-agent RAG pipeline for one question.'''

    retrieve_task = Task(
        description=(
            f"Answer this question about Frankenstein using the search_frankenstein tool.\n\n"
            f"Question: {question}\n\n"
            f"Search the text, then draft an answer. Include direct quotes from the retrieved passages."
        ),
        expected_output="A draft answer supported by direct quotes from Frankenstein.",
        agent=retriever_agent,
    )

    verify_task = Task(
        description=(
            "Review the draft answer from the Literary Researcher.\n\n"
            "For each claim:\n"
            "- If it is directly supported by the quoted passages, keep it.\n"
            "- If it is not supported, remove or correct it.\n"
            "- If the answer cannot be supported by the text, reply 'The text does not specify.'\n\n"
            "Output only the final verified answer — no preamble."
        ),
        expected_output="The final verified answer, grounded only in the text.",
        agent=verifier_agent,
        context=[retrieve_task],
    )

    crew = Crew(
        agents=[retriever_agent, verifier_agent],
        tasks=[retrieve_task, verify_task],
        process=Process.sequential,
        verbose=False,
    )

    result = crew.kickoff()
    return str(result)

# Test on the classic question
q = "Does the monster have a name?"
print(f"Q: {q}\n")
print(f"A: {run_agentic_rag(q)}")

Q: Does the monster have a name?



╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Literary Researcher                                                                                     │
│                                                                                                                 │
│  Task: Answer this question about Frankenstein using the search_frankenstein tool.                              │
│                                                                                                                 │
│  Question: Does the monster have a name?                                                                        │
│                                                                                                                 │
│  Search the text, then draft an answer. Include direct quotes from the retrieved passages.                      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── 🔧 Agent Tool Execution ────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Literary Researcher                                                                                     │
│                                                                                                                 │
│  Thought: Thought: I need to search the text of *Frankenstein* to determine if the monster has a name and to    │
│  find direct quotes to support the answer.                                                                      │
│                                                                                                                 │
│  Using Tool: search_frankenstein                                                                                │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────── Tool Input ───────────────────────────────────────────────────╮
│                                                                                                                 │
│  "{\"query\": \"Does the monster have a name?\"}"                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────── Tool Output ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  His tale is connected and told with an appearance of the simplest truth,                                       │
│  yet I own to you that the letters of Felix and Safie, which he showed me,                                      │
│  and the apparition of the monster seen from our ship, brought to me a                                          │
│  greater conviction of the truth of his narrative than his asseverations,                                       │
│  however earnest and connected. Such a monster has, then, really existence!                                     │
│  I cannot doubt it, yet I am lost in surprise and admiration. Sometimes I                                       │
│  endeavoured to gain from Frankenstein the particulars of his                                                   │
│  creature’s formation, but on this point he was impenetrable.                                                   │
│                                                                                                                 │
│  “Are you mad, my friend?” said he. “Or whither does your                                                       │
│  senseless curiosity lead you? Would you also create for yourself and the                                       │
│  world a demoniacal enemy? Peace, peace! Learn my miseries and do not seek                                      │
│  to increase your own.”                                                                                         │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  yellow light of the moon, as it forced its way through the window                                              │
│  shutters, I beheld the wretch—the miserable monster whom I had                                                 │
│  created. He held up the curtain of the bed; and his eyes, if eyes they                                         │
│  may be called, were fixed on me. His jaws opened, and he muttered some                                         │
│  inarticulate sounds, while a grin wrinkled his cheeks. He might have                                           │
│  spoken, but I did not hear; one hand was stretched out, seemingly to                                           │
│  detain me, but I escaped and rushed downstairs. I took refuge in the                                           │
│  courtyard belonging to the house which I inhabited, where I remained                                           │
│  during the rest of the night, walking up and down in the greatest                                              │
│  agitation, listening attentively, catching and fearing each sound as if                                        │
│  it were to announce the approach of the demoniacal corpse to which I                                           │
│  had so miserably given life.                                                                                   │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  them; sometimes it was the watery, clouded eyes of the monster, as I                                           │
│  first saw them in my chamber at Ingolstadt.          

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Literary Researcher                                                                                     │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  The text does not provide a specific name for the monster, but refers to him as "the monster" or "the          │
│  wretch." For instance, the narrator refers to the creature as:                                                 │
│                                                                                                                 │
│  "I beheld the wretch—the miserable monster whom I had created."                                                │
│                                                                                                                 │
│  The text also refers to him simply as "the monster" when describing his eyes:                                  │
│                                                                                                                 │
│  "them; sometimes it was the watery, clouded eyes of the monster, as I first saw them in my chamber at          │
│  Ingolstadt."                                                                                                   │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Fact Verifier                                                                                           │
│                                                                                                                 │
│  Task: Review the draft answer from the Literary Researcher.                                                    │
│                                                                                                                 │
│  For each claim:                                                                                                │
│  - If it is directly supported by the quoted passages, keep it.                                                 │
│  - If it is not supported, remove or correct it.                                                                │
│  - If the answer cannot be supported by the text, reply 'The text does not specify.'                            │
│                                                                                                                 │
│  Output only the final verified answer — no preamble.                                                           │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Fact Verifier                                                                                           │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  The text does not provide a specific name for the monster, but refers to him as "the monster" or "the          │
│  wretch." For instance, the narrator refers to the creature as: "I beheld the wretch—the miserable monster      │
│  whom I had created." The text also refers to him simply as "the monster" when describing his eyes: "them;      │
│  sometimes it was the watery, clouded eyes of the monster, as I first saw them in my chamber at Ingolstadt."    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── Execution Traces ────────────────────────────────────────────────╮
│                                                                                                                 │
│  🔍 Detailed execution traces are available!                                                                    │
│                                                                                                                 │
│  View insights including:                                                                                       │
│    • Agent decision-making process                                                                              │
│    • Task execution flow and timing                                                                             │
│    • Tool usage details                                                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Would you like to view your execution traces? [y/N] (20s timeout): N
A: The text does not provide a specific name for the monster, but refers to him as "the monster" or "the wretch." For instance, the narrator refers to the creature as: "I beheld the wretch—the miserable monster whom I had created." The text also refers to him simply as "the monster" when describing his eyes: "them; sometimes it was the watery, clouded eyes of the monster, as I first saw them in my chamber at Ingolstadt."


💡 **Notice the extra time.** Agentic RAG is 3-5× slower than simple RAG because we're running multiple LLM calls in sequence. In exchange, we get a verification pass that often catches subtle hallucinations.

**When is it worth it?**
- ✅ High-stakes questions (medical, legal, financial)
- ✅ Multi-hop reasoning where one retrieval isn't enough
- ❌ Simple lookups or chat over small corpora

---

## Section 6 — Gradio chat UI

Wrap it all in a chat interface. Three lines of Gradio and we have a shareable app — `share=True` gives you a public URL that works for ~72 hours.

In [ ]:
# gradio was already installed in Section 0. If you hit import errors, uncomment the line below.
# %pip install -q --upgrade gradio

In [24]:
import gradio as gr

def chat_fn(message, history):
    """Generator — yields partial responses as tokens stream in."""
    partial = ""
    for chunk in simple_rag_chain.stream(message):
        partial += chunk
        yield partial

demo = gr.ChatInterface(
    fn=chat_fn,
    title="💀 Ask Frankenstein",
    description="A RAG-powered chatbot trained on Mary Shelley's 1818 novel.",
    examples=[
        "Who rescues Victor from the Arctic?",
        "What books does the creature read?",
        "Does the monster have a name?",
    ],
    # type="messages",
)

demo.launch(share=True, debug=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://695d092c144beb8700.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


👆 Click the public link (ending in `.gradio.live`) to open the chat in a new tab. Try asking off-topic questions too — see if it sticks to the book.

---

## Section 7 — Head-to-Head: Plain LLM vs Simple RAG vs Agentic RAG 🏁

This is the finale. We ask the same questions to all three systems and compare answers side-by-side.

**Watch for:**
- 🎯 Which system hallucinates on *"monster's name"*?
- 🚫 Which system refuses *"capital of Switzerland"* (correctly, since it's out of scope)?
- ⏱️ How much longer does Agentic RAG take?

In [ ]:
# Three approaches, same interface
def plain_llm(q):
    return ask_plain_llm(q, model='llama3.2:3b')

def simple_rag(q):
    return simple_rag_chain.invoke(q)

def agentic_rag(q):
    return run_agentic_rag(q)

TEST_QUESTIONS = [
    "Does the monster have a name?",                        # Classic hallucination trap
    "Who is the captain of the ship that rescues Victor?",  # Factual, should be easy
    "What is the capital of Switzerland?",                  # Out of scope — should refuse
    "What books does the creature find and read?",          # Specific factual detail
]

In [ ]:
import time
import pandas as pd
from IPython.display import display, HTML

def run_and_time(fn, q, max_chars=500):
    t0 = time.time()
    try:
        answer = fn(q)
    except Exception as e:
        answer = f"[Error: {e}]"
    elapsed = time.time() - t0
    return f"⏱️ {elapsed:.1f}s\n\n{str(answer)[:max_chars]}"

results = []
for q in TEST_QUESTIONS:
    print(f"Running: {q[:70]}...")
    results.append({
        "Question":    q,
        "Plain LLM":   run_and_time(plain_llm, q),
        "Simple RAG":  run_and_time(simple_rag, q),
        "Agentic RAG": run_and_time(agentic_rag, q),
    })
    print("  ✓ done")

df = pd.DataFrame(results)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)

# Render as a wide HTML table for clarity
styled = df.style.set_properties(**{
    'text-align': 'left',
    'vertical-align': 'top',
    'white-space': 'pre-wrap',
    'font-size': '12px',
    'padding': '8px',
    'border': '1px solid #ddd',
}).set_table_styles([
    {'selector': 'th', 'props': [('background', '#f0f0f0'), ('font-weight', 'bold'), ('text-align', 'left')]}
])
display(styled)

### What typically happens

| Question | Plain LLM | Simple RAG | Agentic RAG |
|---|---|---|---|
| **Monster's name** | Often wrong ("Frankenstein") | "The text does not specify" | Same + verified |
| **Ship captain** | Sometimes right (Walton), sometimes hallucinates | Correct + grounded in letters | Correct + cross-checked |
| **Capital of Switzerland** | Answers "Bern" — **out of scope!** | Refuses (good!) | Refuses (good!) |
| **Books the creature reads** | Vague, may invent titles | Specific: *Paradise Lost*, *Plutarch's Lives*, *Sorrows of Werter* | Same + verified |

### Key takeaways

- **Plain LLM** — fastest, but dangerous: confident hallucinations + can't stay in scope
- **Simple RAG** — grounded, refuses to speculate, ~2-5× slower than plain
- **Agentic RAG** — most robust, catches errors the single-pass misses, 3-5× slower still

**No single approach is "best".** Pick based on:
- Accuracy requirements
- Latency budget
- Cost per query (more agents = more LLM calls)
- Question complexity (multi-hop → agentic wins; simple lookup → simple RAG is fine)

---

## Section 8 — Evaluation (bonus)

In production, we don't eyeball answers — we score them against metrics.
**Ragas** is the standard open-source library for this.

### Four core RAG metrics

| Metric | What it measures |
|---|---|
| **Faithfulness** | Is every claim in the answer supported by the retrieved context? |
| **Answer Relevancy** | Does the answer actually address the question? |
| **Context Precision** | Are the retrieved chunks actually useful (no noise)? |
| **Context Recall** | Did we retrieve everything we needed? |

We won't run Ragas live (it's slow — each metric needs multiple LLM calls per example), but here's the pattern for offline evaluation:

In [ ]:
# Sketch — copy this into a separate notebook for offline evaluation.
# Ragas + datasets aren't installed in Section 0. Run the install line first:
#   !pip install -q ragas datasets
# Running live in class would add ~5 minutes.

RAGAS_SKETCH = '''
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision

# Build an eval dataset: questions, our answers, the retrieved contexts, ground-truth answers
eval_data = Dataset.from_dict({
    "question":      ["Does the monster have a name?"],
    "answer":        ["The text does not give the monster a name..."],
    "contexts":      [["<chunk 1>", "<chunk 2>"]],
    "ground_truth":  ["The creature is never named in the novel."],
})

scores = evaluate(
    eval_data,
    metrics=[faithfulness, answer_relevancy, context_precision],
    llm=llm,
    embeddings=embeddings,
)
print(scores)
'''
print(RAGAS_SKETCH)

---

## Section 9 — Takeaways & Next Steps

### What you built today

| # | Component | Lines of code |
|---|---|---|
| 1 | Local LLM chat (Ollama) | ~5 |
| 2 | Chunking pipeline | ~10 |
| 3 | Embedding + vector store | ~8 |
| 4 | Simple RAG chain | ~15 |
| 5 | Agentic RAG (CrewAI) | ~40 |
| 6 | Gradio UI | ~10 |
| 7 | Comparison harness | ~20 |

**Under 120 lines for a working agentic RAG system.** That's the power of modern tooling.

### Where to go next

- 📄 **Swap the document**: replace `frankenstein.txt` with your research PDF, a product manual, your lab's knowledge base
- 🧠 **More agents**: add a Query Decomposer (splits multi-hop questions), a Ranker (reorders chunks), a Summarizer
- 🗄️ **Production vector DB**: ChromaDB → Pinecone, Weaviate, Qdrant, or Postgres + pgvector
- 🔍 **Hybrid search**: combine dense (embeddings) with sparse (BM25) for keyword-heavy corpora
- 📊 **Reranking**: use `bge-reranker` to score query-chunk pairs more precisely after retrieval
- 📡 **Production observability**: LangSmith or Langfuse for tracing + evaluation
- 🛡️ **Guardrails**: structured output, PII scrubbing, jailbreak detection (NeMo Guardrails, Guardrails AI)

### Resources

- Workshop page — https://hakanotal.github.io/workshop-develop-rag-chatbot
- Source repo — https://github.com/hakanotal/workshop-develop-rag-chatbot
- LangChain docs — https://python.langchain.com
- CrewAI docs — https://docs.crewai.com
- Ragas paper — https://arxiv.org/abs/2309.15217
- Ollama model library — https://ollama.com/library

### Questions?

📧 **Hakan Tugrul Otal** — hakan.otal@cyrisk.com
🏛️ Department of Information Sciences & Technology, UAlbany

**Thanks for coming!** 🙏